In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2
    ,min,
    sum
    ,max
    ,coalesce,
    col
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_restructuring_external(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Restructuring"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )
        else:
            logging.info(
                "Tables does not exist, no need to delete entries for business date!"
            )

        # restrloading
        restrloading_extra_filters = None

        if restrloading_extra_filters is not None:
            restrloading_extra_filters.append(date_filter)
            restrloading_final_filters = " AND ".join(restrloading_extra_filters)
        else:
            restrloading_final_filters = date_filter

        restrloading_selected_fields = [
            "AgreementNumber",
            "RestructuredDate",
            "UnitCodeProfits",
            "ModificationAmount",
            "RestructureNumber",
            "IsManualFlag",
            "IsValidatedFlag",
            "AccountNumberOld",
            "AccountNumberNew"
            ,"BusinessDate"
        ]

        restrloading = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_restrloading",
            restrloading_final_filters,
            restrloading_selected_fields,
        ).alias("restrloading")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        ).alias("portfolio")

        # loan_details

        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = [
            "ACCOUNT_NUMBER",
            "prft_product_code",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        ).alias("loan_details")

        # loan_product

        loan_product_extra_filters = None

        if loan_product_extra_filters is not None:
            loan_product_extra_filters.append(date_filter)
            loan_product_final_filters = " AND ".join(loan_product_extra_filters)

        else:
            loan_product_final_filters = date_filter

        loan_product_selected_fields = [
            "product_code",
        ]

        loan_product = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_product",
            loan_product_final_filters,
            loan_product_selected_fields,
        ).alias("loan_product")

        # profits_Params

        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "param_type",
            "serial_num",
        ]

        params = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        ).alias("profits_params")

        # PRE JOIN WORK
        RestructuringExt_join_first_filt = (
            restrloading.where(
                (restrloading["IsManualFlag"] == True)
                & (restrloading["IsValidatedFlag"] == True)
            )
            .groupBy("AccountNumberOld", "AgreementNumber", "RestructureNumber")
            .agg(
                min(restrloading["RestructuredDate"]).alias("NewImplDate"),
                sum(restrloading["ModificationAmount"]).alias("SumModificationAmount"),
            )
            .select(
                "AccountNumberOld" ,
                "AgreementNumber",
                "RestructureNumber",
                "NewImplDate",
                "SumModificationAmount",
            ).withColumnRenamed("AgreementNumber","AgreementNumber_filt")
            .withColumnRenamed("RestructureNumber", "RestructureNumber_filt")
            .withColumnRenamed("AccountNumberOld","AccountNumberOld_filt")
        ).alias("RestructuringExt_join_first_filt")

        RestructuringExt_join_first_filt_b = (
            restrloading.where(
                (restrloading["IsManualFlag"] == True)
                & (restrloading["IsValidatedFlag"] == True)
            )
            .groupBy("AgreementNumber", "RestructureNumber")
            .agg(max(restrloading["ModificationAmount"]).alias("SumModificationAmount"))
            .select("AgreementNumber", "RestructureNumber", "SumModificationAmount")
            .withColumnRenamed("AgreementNumber","AgreementNumber_filt_b")
            .withColumnRenamed("RestructureNumber", "RestructureNumber_filt_b")
        ).alias("RestructuringExt_join_first_filt_b")

        RestructuringExt_join_second_filt = (
            restrloading.where(
                (restrloading["IsManualFlag"] == False)
                & (restrloading["IsValidatedFlag"] == True)
            )
            .groupBy("AccountNumberOld", "AgreementNumber", "RestructureNumber")
            .agg(
                min(restrloading["RestructuredDate"]).alias("NewImplDate"),
                sum(restrloading["ModificationAmount"]).alias("SumModificationAmount"),
            )
            .select(
                "AccountNumberOld",
                "AgreementNumber",
                "RestructureNumber",
                "NewImplDate",
                "SumModificationAmount",
            ).withColumnRenamed("AgreementNumber","AgreementNumber_second_filt")
            .withColumnRenamed("RestructureNumber", "RestructureNumber_second_filt")
            .withColumnRenamed("AccountNumberOld","AccountNumberOld_second_filt")
        ).alias("RestructuringExt_join_second_filt")

      

        # # join conditions

        
        join_condition_1 = (
            
                  (coalesce(restrloading["AccountNumberOld"], lit("1"))
                == coalesce(RestructuringExt_join_first_filt["AccountNumberOld_filt"], lit("1")
                )) &
            
             (
                col("restrloading.AgreementNumber")
                == col("RestructuringExt_join_first_filt.AgreementNumber_filt")
            )
            & (
                col("restrloading.RestructureNumber")
                == col("RestructuringExt_join_first_filt.RestructureNumber_filt")
            )
        )
        join_condition_2 = (
            col("restrloading.AgreementNumber")
            == col("RestructuringExt_join_first_filt_b.AgreementNumber_filt_b")
        ) & (
            col("restrloading.RestructureNumber")
            == col("RestructuringExt_join_first_filt_b.RestructureNumber_filt_b")
        )

        join_condition_3 = (
            (
                coalesce(restrloading["AccountNumberOld"], lit("1"))
                == coalesce(
                    RestructuringExt_join_second_filt["AccountNumberOld_second_filt"], lit("1")
                )
            )
            & (
                restrloading["AgreementNumber"]
                == RestructuringExt_join_second_filt["AgreementNumber_second_filt"]
            )
            & (
                restrloading["RestructureNumber"]
                == RestructuringExt_join_second_filt["RestructureNumber_second_filt"]
            )
        )

        join_condition_4 = (
            restrloading["AgreementNumber"]
            == RestructuringExt_join_first_filt_b["AgreementNumber_filt_b"]
        ) & (
            restrloading["RestructureNumber"]
            == RestructuringExt_join_first_filt_b["RestructureNumber_filt_b"]
        )

        Restructuring_portfolio_join_condition = (
            restrloading["UnitCodeProfits"] == portfolio["unit_code_profits"]
        )
        # # joins
        # # RestructuringExt
        RestructuringExt_join = (
            restrloading.join(
                RestructuringExt_join_first_filt, join_condition_1, "inner"
            )
            .join(RestructuringExt_join_first_filt_b, join_condition_2, "inner")
            .join(
                portfolio,
                Restructuring_portfolio_join_condition,
                "left",
            )
            .join(
                loan_details,
                (restrloading.AccountNumberNew == loan_details.ACCOUNT_NUMBER),
                "left",
            )
            .join(
                loan_product,
                (loan_product.product_code == loan_details.prft_product_code),
                "left",
            )
            .join(
                params,
                (params.param_type == "PRODUCT")
                & (params.serial_num == loan_product.product_code),
                "left",
            )
            .where(
                (restrloading["IsManualFlag"] == True)
                & (restrloading["IsValidatedFlag"] == True)
            )
          
            )
        
        # RestructuringExtProcess
        RestructuringExtProcess_join = (
            restrloading.join(
                RestructuringExt_join_second_filt, join_condition_3, "inner"
            )
            .join(RestructuringExt_join_first_filt_b, join_condition_4, "inner")
            .join(portfolio, Restructuring_portfolio_join_condition, "left")
            .join(
                loan_details,
                (restrloading.AccountNumberNew == loan_details.ACCOUNT_NUMBER),
                "left",
            )
            .join(
                loan_product,
                (loan_product.product_code == loan_details.prft_product_code),
                "left",
            )
            .join(
                params,
                (params.param_type == "PRODUCT")
                & (params.serial_num == loan_product.product_code),
                "left",
            )
            .where(
                (restrloading["IsManualFlag"] == False)
                & (restrloading["IsValidatedFlag"] == True)
            )
        )
        

        # # # selection of columns final!

        restrloading_final_1 = RestructuringExt_join
        

        # # # HashKeys
        result_df_primary_key_fields_1 = [
            restrloading["AgreementNumber"],
            restrloading["AccountNumberNew"],
            restrloading["AccountNumberOld"],
            lit("extfile"),
        ]

        restrloading_final_1 = restrloading_final_1.withColumn(
            "RestructuringId",
            sha2(concat_ws("", *result_df_primary_key_fields_1), 256),
        ).withColumn(
            "RestructuringKey",
            concat_ws(
                ";",
                restrloading["AgreementNumber"],
                restrloading["AccountNumberNew"],
                restrloading["AccountNumberOld"],
            ),
        )
        

        #  # ================== Selection of Final Columns ==================

        result_df_external_1 = restrloading_final_1.select(
            restrloading_final_1.RestructuringId.cast("string"),
            lit("extfile").alias("SourceSystem"),
            portfolio.unit_code_profits.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            restrloading_final_1.RestructuringKey.cast("string").alias("RestructuringKey"),
            restrloading["AgreementNumber"].cast("string").alias("OldAgreementKey"),
            restrloading["AgreementNumber"].cast("string").alias("NewAgreementKey"),
            restrloading["AccountNumberOld"].cast("string").alias("OldExposureKey"),
            restrloading["AccountNumberNew"].cast("string").alias("NewExposureKey"),
            lit(None).alias("WriteOffExposureKey"),
            lit(None).alias("InterestExposureKey"),
            lit(None).alias("InterestFreeExposureKey"),
            lit(None).cast("string").alias("PartyKey"),
            lit(None).cast("string").alias("CDI"),
            lit(None).cast("string").alias("RestructuringProductKey"),
            lit(None).alias("RestructuringLinkType"),
            lit(None).alias("CompletionTargetDate"),
            restrloading.RestructuredDate.cast("date").alias("RestructuringDate"),
            lit(None).cast("string").alias("RestructuringTypeCode"),
            lit(None).cast("string").alias("RestructuringType"),
            RestructuringExt_join_first_filt_b.SumModificationAmount.cast("decimal(19,2)").alias("RestructuringAmount"),
            lit(None).alias("WriteOffAmount"),
            lit(None).alias("RestructuringDiscountAmount"),
            lit(None).alias("SpreadPercentage"),
            lit(None).cast("string").alias("Comment"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("CreatedBy"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            restrloading["BusinessDate"].alias("BusinessDate"),
        )
        restrloading_final_2 = RestructuringExtProcess_join


         # # # HashKeys
        result_df_primary_key_fields_2 = [
            restrloading["AgreementNumber"],
            restrloading["AccountNumberNew"],
            restrloading["AccountNumberOld"],
            lit("extfile"),
        ]

        restrloading_final_2= restrloading_final_2.withColumn(
            "RestructuringId",
            sha2(concat_ws("", *result_df_primary_key_fields_2), 256),
        ).withColumn(
            "RestructuringKey",
            concat_ws(
                ";",
                restrloading["AgreementNumber"],
                restrloading["AccountNumberNew"],
                restrloading["AccountNumberOld"],
            ),
        )
        

        #  # ================== Selection of Final Columns ==================

        result_df_external_2 = restrloading_final_2.select(
            restrloading_final_2.RestructuringId.cast("string"),
            lit("extfile").alias("SourceSystem"),
            portfolio.unit_code_profits.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            restrloading_final_2.RestructuringKey.cast("string").alias("RestructuringKey"),
            restrloading["AgreementNumber"].cast("string").alias("OldAgreementKey"),
            restrloading["AgreementNumber"].cast("string").alias("NewAgreementKey"),
            restrloading["AccountNumberOld"].cast("string").alias("OldExposureKey"),
            restrloading["AccountNumberNew"].cast("string").alias("NewExposureKey"),
            lit(None).alias("WriteOffExposureKey"),
            lit(None).alias("InterestExposureKey"),
            lit(None).alias("InterestFreeExposureKey"),
            lit(None).cast("string").alias("PartyKey"),
            lit(None).cast("string").alias("CDI"),
            lit(None).cast("string").alias("RestructuringProductKey"),
            lit(None).alias("RestructuringLinkType"),
            lit(None).alias("CompletionTargetDate"),
            restrloading.RestructuredDate.cast("date").alias("RestructuringDate"),
            lit(None).cast("string").alias("RestructuringTypeCode"),
            lit(None).cast("string").alias("RestructuringType"),
            RestructuringExt_join_first_filt_b.SumModificationAmount.cast("decimal(19,2)").alias("RestructuringAmount"),
            lit(None).alias("WriteOffAmount"),
            lit(None).alias("RestructuringDiscountAmount"),
            lit(None).alias("SpreadPercentage"),
            lit(None).cast("string").alias("Comment"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("CreatedBy"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            restrloading["BusinessDate"].alias("BusinessDate"),
        )


        result_df_external = result_df_external_1.union(result_df_external_2)

        result_df_external = (
            result_df_external.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_external.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)


    else:
        null_args = [
               arg_name
               for arg_name, arg in args
               if arg
               in [
                   curated_zone,
                   catalog_landed,
                   reference_zone,
                   run_date,
                   schema,
                   env_prefix,
               ]
               and arg is None
           ]   
        for arg_name in null_args:
               logging.error(f"No {arg_name} specified!")   
    logging.info("We did it!")

In [0]:
load_curated_restructuring_external(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)